# 🛡️ Self-Healing RAG Pipeline — Walkthrough & Demonstration

This notebook demonstrates the **Self-Healing RAG** pipeline. Unlike traditional single-pass RAG (which blindly returns generated text regardless of factual support), Self-Healing RAG:
1. **Critiques** generated answers against source context chunks.
2. **Detects hallucinations** and ungrounded claims.
3. **Reformulates queries** to re-retrieve missing context upon rejection.
4. **Falls back gracefully** when max retries are reached without sufficient evidence.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('../src'))

from scripts.ingest import ingest_sample_docs
from selfhealing_rag.config import settings
from selfhealing_rag.critic import Critic
from selfhealing_rag.generator import Generator
from selfhealing_rag.llm_client import AnthropicLLMClient, MockLLMClient
from selfhealing_rag.orchestrator import Orchestrator
from selfhealing_rag.reformulator import QueryReformulator
from selfhealing_rag.retriever import Retriever
from selfhealing_rag.vector_store import ChromaVectorStore

print("Loaded Self-Healing RAG components successfully.")

## Step 1: Ingest Sample Documentation into Vector Database
First, we index our company policy and cloud security standards sample documents into ChromaDB.

In [ ]:
ingest_sample_docs(docs_dir='../data/sample_docs')

## Step 2: Initialize Pipeline Components
We set up the vector store, retriever, generator, critic, query reformulator, and orchestrator.

In [ ]:
llm_client = AnthropicLLMClient() if settings.anthropic_api_key else MockLLMClient()
vector_store = ChromaVectorStore()
retriever = Retriever(vector_store=vector_store)
generator = Generator(llm_client=llm_client)
critic = Critic(llm_client=llm_client)
reformulator = QueryReformulator(llm_client=llm_client)

orchestrator = Orchestrator(
    retriever=retriever,
    generator=generator,
    critic=critic,
    reformulator=reformulator,
    max_retries=2
)
print("Orchestrator ready.")

## Scenario 1: Supported Query (First-Try Success)
Querying information present directly in the knowledge base.

In [ ]:
query_1 = "What encryption standard is required for data in transit and data at rest?"
response_1 = orchestrator.run(query=query_1, enable_self_healing=True)

print(f"Status: {response_1.status}")
print(f"Attempts: {response_1.total_attempts}")
print(f"Answer:\n{response_1.answer}")

## Scenario 2: Out-of-Domain Query (Graceful Fallback)
Querying information NOT in the knowledge base to observe critic rejection and honest fallback.

In [ ]:
query_2 = "What is the mandatory SOC 2 Type II audit frequency for AI models?"
response_2 = orchestrator.run(query=query_2, enable_self_healing=True)

print(f"Status: {response_2.status}")
print(f"Attempts: {response_2.total_attempts}")
print(f"Answer:\n{response_2.answer}")